# TIA and the rule baselines on MLTaskBench (Kaggle, self-contained)

An external check for the paper: the task-inference model TIA and four rule baselines predict the task of the 319
tabular targets of MLTaskBench v2 (Zenodo, CC BY 4.0), from values only (C0), with names (C1) and with documentation (C2).
Scoring happens later on our side; this notebook never reads the MLTaskBench labels.

**Everything is inside this notebook.** No dataset needs to be attached. The 319 tables (about 108 MB of the 4.2 GB
archive) are read straight from Zenodo.

**Settings (right-hand panel):** Accelerator **None** (CPU). **Internet On** (needed for Zenodo and for installing
scikit-learn 1.6.1).

**Run:** Save Version › **Save & Run All (Commit)**, then you can close the browser. About 20–30 minutes. Step 3
stops the run if this environment cannot reproduce TIA's frozen predictions. When the version has finished, open it ›
**Output** › download `mltaskbench_results.zip` and put it in the project folder.

## 1. Unpack the bundle

The first cell holds the embedded code and inputs; the second unpacks them and checks every file against its checksum.

In [ ]:
# Embedded bundle: code and inputs, ~381 kB
PAYLOAD_SHA256 = "60ccabeb60b760acd4e24c35620faae70efbdc51fb3ec8f6037b069c36958392"
FILE_SHA256 = {
 "config.json": "5ecc5dbafddbcbc9ce84177759b96e57c712f092b7d5e24b2762e5c2d212fa88",
 "data_lib.py": "a301af0edd1b9ae25b8dd646c2a9a8c484079136977dbe6cbb258c1f8210f6dd",
 "inputs/features.csv": "a6c9072d3dbead2134ecbd47c7b6e105937a1896736f77c9e6794c78164c2755",
 "inputs/frozen_check.csv": "59ea4191474496497a99ad4dd16048f0a39e17e9b4a16f6f939b7559de194a75",
 "inputs/manifest.csv": "a1dcde336cc003f596d848537b96f5ebd531eb3d9e989f68d745e2dcc78e9fa5",
 "inputs/targets.csv": "14bb21fce75d9cc32b2c541df0e11e9741963d5e5226d73358d46a1e95f05f5a",
 "lexicons.py": "7e2dc5fe1e2aa1cff58a2f7cf6075c17317c929e1c956f50d1bd2ae296ebc7ad",
 "method_lib.py": "4f89d20ff097f6c545ef5c15bc007a21712bb7f9a80c06123213787fa5234d53",
 "mltaskbench_lib.py": "9cfa7db4c012787579fb9ca493233218099fbf47e525c1519c9f0635b404cd3f",
 "robustness_lib.py": "d9f8ed4288c7fce9a6f3332d275d7c510535f1944d48327bdb1d332616a58960",
 "rules_lib.py": "5eb6c90f0c4ae67989078890f1f57891d70c43120b050f0e4cc2340ec7e303f1"
}
PAYLOAD = """
...
rHAAAdi6EYCQWIwiLnqxxGf7AgAAAAAEWVo=
"""

In [ ]:
import base64, hashlib, io, os, tarfile
from pathlib import Path

if Path("/kaggle/working").is_dir():
    WORK, OUT_ZIP = Path("/tmp/mltaskbench"), Path("/kaggle/working/mltaskbench_results.zip")
elif Path("/content").is_dir():
    WORK, OUT_ZIP = Path("/content/mltaskbench"), Path("/content/mltaskbench_results.zip")
else:
    WORK = Path(os.environ.get("MLTB_WORK", Path.cwd() / "mltaskbench_run"))
    OUT_ZIP = WORK.parent / "mltaskbench_results.zip"
raw = base64.b64decode("".join(PAYLOAD.split()))
assert hashlib.sha256(raw).hexdigest() == PAYLOAD_SHA256, "the embedded bundle is damaged: download the notebook again"
for d in (WORK, WORK.parent / "mltaskbench_smoke"):
    d.mkdir(parents=True, exist_ok=True)
    with tarfile.open(fileobj=io.BytesIO(raw), mode="r:xz") as tar:
        if hasattr(tarfile, "data_filter"):
            tar.extractall(d, filter="data")
        else:
            tar.extractall(d)
    for name, sha in FILE_SHA256.items():
        assert hashlib.sha256((d / name).read_bytes()).hexdigest() == sha, f"{name} does not match its checksum"
print(f"unpacked to {WORK}: {len(FILE_SHA256)} files, checksums ok")

## 2. Install scikit-learn 1.6.1 (needs Internet On)

TIA was trained with this version; the next cell refuses to continue with any other.

In [ ]:
%pip install -q "scikit-learn==1.6.1"

In [ ]:
import importlib, json, sys
assert "WORK" in globals(), "run the cells from the top"
if str(WORK) not in sys.path:
    sys.path.insert(0, str(WORK))
import mltaskbench_lib as mt
CFG, TARGETS = mt.load(WORK)
mt.check_sklearn(CFG)
print(len(TARGETS), "targets;", mt.versions())

## 3. Smoke test (about a minute)

Three small tables, a reduced training set, nothing kept. It checks that Zenodo can be reached and that the code runs here.

In [ ]:
smoke = mt.run(WORK.parent / "mltaskbench_smoke", only=set(CFG["smoke_ids"]), smoke=True)
assert smoke["read_ok"] == len(CFG["smoke_ids"]), "the smoke test could not read its tables: stop and send the output"
print("smoke test ok")

## 4. The run (about 20 minutes)

Reads the 319 tables, builds the inputs, runs the rules, retrains TIA and checks it against its frozen predictions (the run stops here if they differ), then predicts. The results zip is written after the rules and again at the end.

In [ ]:
report = mt.run(WORK, dest_zip=OUT_ZIP)
print(json.dumps({k: report[k] for k in ("fetch", "read_ok", "read_failed", "fidelity_passed", "seconds")}, indent=1, default=str))
print("wrote", OUT_ZIP)